<a href="https://colab.research.google.com/github/sairas2124/Gradient_descent-/blob/main/health_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("heart_disease_uci.csv")

df = df.drop(['id','dataset', 'ca', 'thal'], axis=1)
# -------------------------------
# Step 1: Make a copy
# -------------------------------
df_sd = df.copy()

# -------------------------------
# Step 2: Fix column names (if needed)
# -------------------------------
df_sd.rename(columns={
    'num': 'target'
}, inplace=True)

# -------------------------------
# Step 3: Convert target to binary (if needed)
# -------------------------------
if 'target' in df_sd.columns:
    df_sd['target'] = df_sd['target'].apply(lambda x: 1 if x > 0 else 0)

# -------------------------------
# Step 4: Identify columns
# -------------------------------
# Numerical columns
num_cols = df_sd.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Remove target from numerical columns
if 'target' in num_cols:
    num_cols.remove('target')

# Categorical columns
cat_cols = df_sd.select_dtypes(include=['object']).columns.tolist()

# -------------------------------
# Step 5: Fill numerical with (mean + std)
# -------------------------------
for col in num_cols:
    mean_val = df_sd[col].mean()
    std_val = df_sd[col].std()

    fill_value = mean_val + std_val

    df_sd[col] = df_sd[col].fillna(fill_value)

# -------------------------------
# Step 6: Fill categorical with mode
# -------------------------------
for col in cat_cols:
    df_sd[col] = df_sd[col].fillna(df_sd[col].mode()[0])


# -------------------------------
# Convert categorical to numeric
# -------------------------------
df['sex'] = df['sex'].map({'Male': 1, 'Female': 0})

print("Missing values after cleaning:\n")
print(df_sd.isnull().sum())

df_sd.to_csv("heart_cleaned_sd.csv", index=False)

print("\n✅ Dataset cleaned and saved as heart_cleaned_sd.csv")

Missing values after cleaning:

age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalch      0
exang       0
oldpeak     0
slope       0
target      0
dtype: int64

✅ Dataset cleaned and saved as heart_cleaned_sd.csv


/tmp/ipykernel_4223/3953857985.py:54: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sd[col] = df_sd[col].fillna(df_sd[col].mode()[0])


In [17]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd # Import pandas for get_dummies

# Define X (features) and y (target)
y = df_sd['target']
X = df_sd.drop('target', axis=1)

# One-hot encode categorical features
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42
)

# Scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to tensor
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [18]:
import torch.nn as nn

class HeartNN(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_size, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [19]:
model = HeartNN(X_train.shape[1])

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):

    # Forward
    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 0.6994001865386963
Epoch 10, Loss: 0.5973082780838013
Epoch 20, Loss: 0.4652220606803894
Epoch 30, Loss: 0.4041506350040436
Epoch 40, Loss: 0.38377633690834045
Epoch 50, Loss: 0.3691621720790863
Epoch 60, Loss: 0.3582322895526886
Epoch 70, Loss: 0.3468828499317169
Epoch 80, Loss: 0.3344123959541321
Epoch 90, Loss: 0.3193732500076294


In [20]:
# Save model
torch.save(model.state_dict(), "heart_nn.pth")

print("✅ PyTorch model saved")

import pickle

# Save column structure
pickle.dump(X.columns.tolist(), open("model_columns.pkl", "wb"))

print("✅ Columns saved")

✅ PyTorch model saved
✅ Columns saved


In [21]:
with torch.no_grad():
    preds = model(X_test)
    preds = (preds > 0.5).float()

    accuracy = (preds == y_test).float().mean()

print("PyTorch Accuracy:", accuracy.item())

PyTorch Accuracy: 0.83152174949646


In [22]:
import pickle

pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(X.columns.tolist(), open("model_columns.pkl", "wb"))

print("✅ Scaler and columns saved")

✅ Scaler and columns saved
